# Proj2 — Light Full-Data 2-Config Sweep

This notebook is a low-memory version of the project sweep for DSMLP MIG.

**What it does:**
- Uses full `train.json` + `validation.json`
- Runs only **2 configs** (0.5B wide and 1.5B wide)
- Enables **QLoRA** to reduce VRAM pressure
- Uses conservative training args (`batch=1`, `max_length=2048`, `num_chunks=1`)

| Catalog ID | Model | LR | LoRA |
|------------|-------|-----|------|
| 0 | Qwen2.5-0.5B-Instruct | 5e-5 | wide (r=16) |
| 1 | Qwen2.5-1.5B-Instruct | 5e-5 | wide (r=16) |

RapidFire assigns internal run ids `1..N` for selected configs.

## 0. Debug knobs (edit here)

In [1]:
from datetime import datetime

from paths import LOGS_DIR, REPO_ROOT, TRAIN_JSON, VAL_JSON, print_paths, require_release_data, setup

setup(chdir_try=True)  # cwd → try/; ./logs → try/logs; imports schema_*.py from here
require_release_data()
print_paths()

# --- What to run (catalog ids) ---
RUN_IDS = [0, 1]    # 0: 0.5B narrow, 1: 1.5B narrow

# --- Full-data but light training ---
NUM_EPOCHS = 1
MAX_LENGTH = 2048 # increase this for less truncation after step 2
USE_QLORA = False # set this to reduce issues 

# --- Experiment naming (timestamp avoids RapidFire suffix churn) ---
EXPERIMENT_BASE = "proj2-debug-2config"
EXPERIMENT_NAME = f"{EXPERIMENT_BASE}-{datetime.now():%Y%m%d-%H%M}"
EXPERIMENT_PATH = "./logs"

# --- RapidFire scheduling ---
NUM_CHUNKS = 1      # reduce allocator fragmentation (worker reset per chunk)
NUM_GPUS = 1
SEED = 42

SCHEMA_STYLE = "minimal"         # or "full"

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"RUN_IDS={RUN_IDS}  epochs={NUM_EPOCHS}  max_length={MAX_LENGTH}  qlora={USE_QLORA}")
print(f"NUM_CHUNKS={NUM_CHUNKS}  (full train + val)")

REPO_ROOT   = /home/emc001/schema-linker-sft-pipeline
TRY_DIR     = /home/emc001/schema-linker-sft-pipeline/try
SCHEMAS_DIR = /home/emc001/schema-linker-sft-pipeline/schemas  (18 json files)
LOGS_DIR    = /home/emc001/schema-linker-sft-pipeline/try/logs
RF_HOME     = /home/emc001/rapidfireai
cwd         = /home/emc001/schema-linker-sft-pipeline/try
Experiment: proj2-debug-2config-20260530-1216
RUN_IDS=[0, 1]  epochs=1  max_length=2048  qlora=False
NUM_CHUNKS=1  (full train + val)


## 1. Optional — free GPU / end stale experiment

In [2]:
import gc

def cuda_cleanup():
  """Call between failed runs to reduce allocator/NVML pressure."""
  gc.collect()
  try:
    import torch
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
      torch.cuda.synchronize()
      print("CUDA cache cleared")
  except Exception as e:
    print(f"cuda_cleanup skipped: {e}")

# Uncomment if a previous Experiment object is still in memory:
# experiment.end()
# cuda_cleanup()

## 2. Imports & data

In [3]:
import json
from functools import lru_cache

from datasets import Dataset
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

from schema_formatting import make_formatting_func
from schema_metrics import make_compute_metrics

# TRAIN_JSON / VAL_JSON set in section 0 via paths.setup()
with open(TRAIN_JSON) as f:
    raw_train = json.load(f)
with open(VAL_JSON) as f:
    raw_val = json.load(f)

train_ds = Dataset.from_list(raw_train)
val_ds = Dataset.from_list(raw_val)
train_ds = train_ds.shuffle(seed=SEED)
fmt_func = make_formatting_func(SCHEMA_STYLE)

print(f"Train={len(train_ds)}  Val={len(val_ds)}  schema={SCHEMA_STYLE}")

Train=301  Val=101  schema=minimal


## 3. Model factory & config grid (2 lightweight runs)

In [ ]:
@lru_cache
def get_tokenizer(model_name: str):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def create_model(model_config: dict):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_kwargs = model_config["model_kwargs"].copy()
    use_qlora = model_kwargs.pop("use_qlora", False)

    if use_qlora:
        from peft import prepare_model_for_kbit_training
        from transformers import BitsAndBytesConfig
        import torch
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_cfg, **model_kwargs
        )
        model.enable_input_require_grads()   # required for QLoRA + gradient_checkpointing
        
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    return (model, tokenizer)


peft_narrow = RFLoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
def make_sft_args(lr: float) -> RFSFTConfig:
    return RFSFTConfig(
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05, # added warm up loss keeps exploding!
        max_grad_norm=0.3, # cap the grad
        weight_decay=0.01, # prevent from memorizing
        completion_only_loss=True, #  mask prompt, only train on JSON target
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=NUM_EPOCHS,
        max_length=MAX_LENGTH,
        logging_steps=10,
        eval_accumulation_steps=1, # offload eval preds to CPU per batch
        eval_strategy="epoch",
        load_best_model_at_end=False,
        gradient_checkpointing=True,
        # gradient_checkpointing=False, # # caused an error for QLoRA
        bf16=True,
        fp16=False,
        report_to="none",
    )


def model_kwargs() -> dict:
    kw = {"device_map": "auto", "torch_dtype": "auto", "use_cache": False}
    if USE_QLORA:
        kw["use_qlora"] = True
    return kw


GEN_CFG = {"max_new_tokens": 512, "do_sample": False, "temperature": 1.0}

QWEN_05B = "Qwen/Qwen2.5-0.5B-Instruct"
QWEN_15B = "Qwen/Qwen2.5-1.5B-Instruct"

RUN_CATALOG = [
    {"run_id": 0, "model": QWEN_05B, "lr": 2e-5, "label": "0.5B | lr=2e-5 | wider"}, # make lr smaller
    {"run_id": 1, "model": QWEN_15B, "lr": 2e-5, "label": "1.5B | lr=2e-5 | wider"},
]


def catalog_entry_to_rf_config(entry: dict) -> RFModelConfig:
    return RFModelConfig(
        model_name=entry["model"],
        peft_config=peft_narrow,
        training_args=make_sft_args(entry["lr"]),
        model_type="causal_lm",
        model_kwargs=model_kwargs(),
        formatting_func=fmt_func,
        # compute_metrics=compute_metrics_for_rf, # remote this -> causing the 0.0149 score
        compute_metrics=make_compute_metrics(get_tokenizer(entry["model"])),
        # generation_config=GEN_CFG.copy() # removing this bc of leaderboard scores
    )


selected = [e for e in RUN_CATALOG if e["run_id"] in RUN_IDS]
if not selected:
    raise ValueError(f"RUN_IDS {RUN_IDS} matched no runs")

for i, e in enumerate(selected, start=1):
    print(f"  catalog {e['run_id']} -> run ~{i}: {e['label']}  qlora={USE_QLORA}")

config_set = List([catalog_entry_to_rf_config(e) for e in selected])
config_group = RFGridSearch(configs=config_set, trainer_type="SFT")

print(f"\nLaunching {len(selected)} run(s) on {len(train_ds)} train / {len(val_ds)} val examples")

  catalog 0 -> run ~1: 0.5B | lr=2e-5 | wider  qlora=False
  catalog 1 -> run ~2: 1.5B | lr=2e-5 | wider  qlora=False

Launching 2 run(s) on 301 train / 101 val examples


## 4. Create experiment & run

In [5]:
cuda_cleanup()

experiment = Experiment(
    experiment_name=EXPERIMENT_NAME,
    mode="fit",
    experiment_path=EXPERIMENT_PATH,
)
print(f"Logs: {EXPERIMENT_PATH}/{experiment.experiment_name}")

CUDA cache cleared
Experiment proj2-debug-2config-20260530-1216 created with Experiment ID: 26 and Metric Experiment ID: 25 at /home/emc001/schema-linker-sft-pipeline/try/logs/proj2-debug-2config-20260530-1216
Logs: ./logs/proj2-debug-2config-20260530-1216


In [6]:
cfg = catalog_entry_to_rf_config(selected[0])
print(cfg.training_args.learning_rate)
print(cfg.training_args.warmup_ratio)
print(cfg.training_args.max_grad_norm)

2e-05
0.05
0.3


In [7]:
# Full-data light run (2 configs). If OOM persists: lower MAX_LENGTH or NUM_EPOCHS in section 0.
experiment.run_fit(
    config_group,
    create_model,
    train_ds,
    val_ds,
    num_chunks=NUM_CHUNKS,
    seed=SEED,
    num_gpus=NUM_GPUS,
)

INFO 05-30 12:17:07 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-30 12:17:23 [__init__.py:216] Automatically detected platform cuda.
Run 2 has failed: NVML_SUCCESS == r INTERNAL ASSERT FAILED at "/pytorch/c10/cuda/CUDACachingAllocator.cpp":1131, please report a bug to PyTorch. Traceback (most recent call last):
  File "/opt/conda/envs/cse234/lib/python3.12/site-packages/rapidfireai/fit/backend/worker.py", line 676, in serve_forever
    self.run_fit(
  File "/opt/conda/envs/cse234/lib/python3.12/site-packages/rapidfireai/fit/backend/worker.py", line 387, in run_fit
    trainer_instance.train()
  File "/opt/conda/envs/cse234/lib/python3.12/site-packages/transformers/trainer.py", line 2325, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/cse234/lib/python3.12/site-packages/transformers/trainer.py", line 2790, in _inner_training_loop
    self._maybe_log_save_evaluate(

## 5. Results & cleanup

In [8]:
results = experiment.get_results()
display(results)

,run_id,step,loss,grad_norm,learning_rate,num_tokens,mean_token_accuracy,chunk number,num_epochs_completed
0,1,10,1.9528,1.064692,1.976296e-05,80992.0,0.366015,0.0,0.0
1,1,20,1.9033,0.987793,1.793353e-05,160853.0,0.368272,0.0,0.0
2,1,30,1.8441,0.936219,1.461749e-05,240209.0,0.268771,0.0,0.0
3,1,40,2.3739,0.834776,1.043619e-05,317838.0,0.356986,0.0,0.0
4,1,50,1.7610,0.787554,6.173166e-06,396894.0,0.408403,0.0,0.0
5,1,60,1.7332,0.841129,2.627227e-06,476682.0,0.428877,0.0,0.0
6,1,70,1.7205,0.937376,4.628305e-07,556798.0,0.292546,0.0,0.0
7,2,10,1.7702,0.472420,1.976296e-05,80992.0,0.370849,0.0,0.0
8,2,20,1.7357,0.457235,1.793353e-05,160853.0,0.373070,0.0,0.0
9,2,30,1.6922,0.479696,1.461749e-05,240209.0,0.272185,0.0,0.0


In [9]:
experiment.end()
cuda_cleanup()
print("Experiment ended.")

Experiment proj2-debug-2config-20260530-1216 ended
Workers stopped
CUDA cache cleared
Experiment ended.


## Plot

In [10]:
import subprocess, json, re

def run_eval(preds_path, run_label, results_file="eval_results.json",
            gold="validation_gold_schema_links.json",
            questions="validation_input.json",
            schemas_dir="schemas/"):
    result = subprocess.run(
        ["python", "eval.py",
        "--predictions", preds_path,
        "--gold", gold,
        "--schemas_dir", schemas_dir,
        "--questions_input", questions],
        capture_output=True, text=True,
        cwd="/home/emc001/schema-linker-sft-pipeline"
    )
    print(result.stdout)

    def _f(pattern):
        m = re.search(pattern, result.stdout)
        return float(m.group(1)) if m else None

    metrics = {
        "run": run_label,
        "table_P": _f(r"Precision_T\s*:\s*([0-9.]+)"),
        "table_R": _f(r"Recall_T\s*:\s*([0-9.]+)"),
        "table_F1": _f(r"F1_T\s*:\s*([0-9.]+)"),
        "col_P": _f(r"Precision_C\s*:\s*([0-9.]+)"),
        "col_R": _f(r"Recall_C\s*:\s*([0-9.]+)"),
        "col_F1": _f(r"F1_C\s*:\s*([0-9.]+)"),
        "table_score": _f(r"Table Score\s*:\s*([0-9.]+)"),
        "col_score": _f(r"Column Score\s*:\s*([0-9.]+)"),
        "leaderboard": _f(r"Leaderboard Score\s*:\s*([0-9.]+)"),
    }
    try:
        with open(results_file) as f:
            all_results = json.load(f)
    except FileNotFoundError:
        all_results = []
    all_results.append(metrics)
    with open(results_file, "w") as f:
        json.dump(all_results, f, indent=2)
    return metrics

In [11]:
df = pd.read_json("eval_results.json")
    
# Example calls:
run_eval("preds_run1.json", run_label="0.5B lr=2e-5")
run_eval("preds_run2.json", run_label="1.5B lr=2e-5")


# Cell — Plot:

import pandas as pd, matplotlib.pyplot as plt


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
metrics_groups = [
    ("Table P/R/F1", ["table_P", "table_R", "table_F1"]),
    ("Scores",       ["table_score", "col_score", "leaderboard"]),
]
for ax, (title, cols) in zip(axes, metrics_groups):
    df.set_index("run")[cols].plot(kind="bar", ax=ax, ylim=(0, 1))
    ax.set_title(title)
    ax.set_xticklabels(df["run"], rotation=15, ha="right")
    ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.savefig("eval_comparison.png", dpi=120)
plt.show()

NameError: name 'pd' is not defined